# Project: Plan Your Trip with Kayak & Weather Data

## Objective
The goal of this project is to recommend the **top 5 cities in France** to visit and the **top 20 hotels** in that area, based on the best weather forecast for the coming days.

## Workflow
1.  **Geolocation**: Retrieve GPS coordinates (latitude, longitude) for a list of target cities using the Nominatim API.
2.  **Weather Forecast**: Fetch 5-day weather forecasts for each city using the OpenWeatherMap API.
3.  **Scoring**: Calculate a "Weather Score" for each city based on temperature, rain probability, wind, etc., to identify the best destinations.
4.  **Accommodation**: Scrape Booking.com for the top hotels in each city.
5.  **Storage**: Save the cleaned data into a **Cloudflare R2** bucket (data lake) and a **PostgreSQL database** (data warehouse) for further analysis or visualization.

> The forecast covers **5 days**, not the 7 mentioned in the assignment: OpenWeatherMap has since moved its 7-day endpoint out of the free plan. See the README for this and the other documented deviations.

In [ ]:
# Import libraries
import pandas as pd
import requests
import json
import time
import datetime
import uuid
import os
import plotly.express as px
from dotenv import load_dotenv
import boto3
from sqlalchemy import create_engine, text, Table, Column, Integer, String, MetaData, ForeignKey, Float, Date

# Local convenience only. Credentials live in a .env that git never sees; see
# .env.example for the list of keys. Nothing below may depend on that file
# existing -- exporting the same variables works just as well.
load_dotenv()

# Configuration Variables
weather_api_secret = os.getenv('WEATHER_API_SECRET')

# Data lake -- Cloudflare R2. It implements the S3 API, so boto3 needs no plugin
# and no rewrite: the endpoint below is the only thing pointing the calls away
# from AWS.
r2_access_key_id = os.getenv('R2_ACCESS_KEY_ID')
r2_secret_access_key = os.getenv('R2_SECRET_ACCESS_KEY')
r2_endpoint_url = os.getenv('R2_ENDPOINT_URL')
bucket_name = os.getenv('R2_BUCKET')

# Data warehouse -- managed PostgreSQL. Use the DIRECT endpoint, not the pooled
# one: the pooler runs in transaction mode and mishandles the DDL that
# meta.create_all() issues below.
host = os.getenv('HOST_SQL_ALCHEMY')
port = os.getenv('PORT_SQL_ALCHEMY')
database = os.getenv('DATABASE_SQL_ALCHEMY')
user = os.getenv('USER_SQL_ALCHEMY')
password = os.getenv('PASSWORD_SQL_ALCHEMY')

In [ ]:
# Read initial list of cities from JSON
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

In [ ]:
# Create a working copy to avoid modifying the original dataframe
df = df_source.copy()

In [ ]:
# Define headers for API calls to mimic a real browser user-agent
# This helps avoid being blocked by some APIs.
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

# 1. Geolocation

In [ ]:
# Fetch coordinates for each city using Nominatim API
# We add a delay (time.sleep) to respect the API's usage policy and avoid rate limiting.

for index, row in df.iterrows():
    try:
        # API Call
        url = f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json"
        res = requests.get(url, headers=headers)
        
        if res.status_code == 200 and res.json():
            city_data = res.json()[0]
            df.loc[index, "lat"] = city_data["lat"]
            df.loc[index, "lon"] = city_data["lon"]
        else:
            print(f"Could not find coordinates for {row['city']}")
            
    except Exception as e:
        print(f"Error processing {row['city']}: {e}")
        
    time.sleep(1) # Pause to respect API rate limits

df.head()

# Checkpoint: Save result to CSV to avoid re-running expensive API calls
df.to_csv("cities_with_geoposition.csv", index=False)

In [ ]:
# [Optional] Reload data from CSV if restarting the notebook
if os.path.exists("cities_with_geoposition.csv"):
    df = pd.read_csv("cities_with_geoposition.csv")
    print("Loaded cities data from CSV.")
else:
    print("CSV not found, using data from memory.")
    
df.head()

# 2. Weather Forecast

In [ ]:
# Fetch 5-day/3-hour forecast data from OpenWeatherMap
list_weather_data = []

for index, row in df.iterrows():
    # Check if we have valid coordinates
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue

    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&appid={weather_api_secret}"
    res_weather = requests.get(url, headers=headers)
    res_weather_json = res_weather.json()
    
    # Process each forecast entry in the response
    if 'list' in res_weather_json:
        for res in res_weather_json['list']:
            weather_entry = {
                "city": row['city'],
                "lat": row['lat'],
                "lon": row['lon'],
                "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%Y-%m-%d'),
                "hour": datetime.datetime.fromtimestamp(res['dt']).strftime('%H:%M'),
                "temp": res['main']['temp'],
                "prob_rain": res.get('pop', 0), # Probability of precipitation (0-1)
                # 'rain' is absent from the payload when no rain is expected
                "volume_rain": res.get('rain', {}).get('3h', 0),
                "wind_speed": res['wind']['speed'],
                "perc_cloud": res['clouds']['all']
            }
            list_weather_data.append(weather_entry)
    
    time.sleep(1) # Pause to respect API rate limits

df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())

# Checkpoint: Save weather data
df_weather.to_csv("weather_forecast.csv", index=False)

In [ ]:
# [Optional] Reload weather data
if os.path.exists("weather_forecast.csv"):
    df_weather = pd.read_csv("weather_forecast.csv")
    print("Loaded weather data from CSV.")
df_weather.head()

In [ ]:
# Data Pre-processing
# Convert probability of rain to percentage (0-100)
# Convert wind speed from m/s to km/h
df_weather['prob_rain'] = df_weather['prob_rain'] * 100
df_weather['wind_speed'] = df_weather['wind_speed'] * 3.6
df_weather.head()

In [ ]:
# Aggregate daily data
# We group by city and date to get daily statistics (Mean/Max/Min)
df_weather_groupby = df_weather.groupby(['city', 'lat', 'lon', 'date']).agg({
    'temp': ['mean', 'min', 'max'], 
    'prob_rain': 'max', 
    'volume_rain': ['mean', 'max', 'sum'], 
    'wind_speed': 'max', 
    'perc_cloud': 'mean'
}).reset_index()

df_weather_groupby.head()

## Scoring Methodology

We calculate a satisfaction score (0-100) for each weather metric, where **100 is perfect** and **0 is poor**.

### 1. Normalization (0-100)
| Metric | Target | Penalty Calculation |
| :--- | :--- | :--- |
| **Temperature** | 25°C | -4 pts per degree deviation from 25°C |
| **Rain Probability** | 0% | 100 - (Probability %) |
| **Rain Volume** | 0mm | -5 pts per mm |
| **Wind Speed** | 0 km/h | 100 - (Speed in km/h) |
| **Cloudiness** | 0% | 100 - (Cloud %) |

### 2. Weighted Final Score
The final score is a weighted average of individual scores:
- **Temperature**: 30%
- **Rain Probability**: 20%
- **Rain Volume**: 30% (Heavy penalty for rain)
- **Wind**: 10%
- **Clouds**: 10%

In [ ]:
# 1. Calculate Individual Scores

# Temperature: Target 25°C
df_weather_groupby['score_temp'] = 100 - (abs(df_weather_groupby[('temp', 'max')] - 25) * 4)
df_weather_groupby['score_temp'] = df_weather_groupby['score_temp'].clip(lower=0)

# Rain Probability: 0% is best
df_weather_groupby['score_rain_prob'] = 100 - (df_weather_groupby[('prob_rain', 'max')])

# Rain Volume: 0mm is best
df_weather_groupby['score_rain_vol'] = 100 - (df_weather_groupby[('volume_rain', 'sum')] * 5)
df_weather_groupby['score_rain_vol'] = df_weather_groupby['score_rain_vol'].clip(lower=0)

# Wind Speed: 0 km/h is best
df_weather_groupby['score_wind'] = 100 - df_weather_groupby[('wind_speed', 'max')]
df_weather_groupby['score_wind'] = df_weather_groupby['score_wind'].clip(lower=0)

# Cloudiness: 0% is best
df_weather_groupby['score_cloud'] = 100 - df_weather_groupby[('perc_cloud', 'mean')]

# 2. Calculate Weighted Final Score
df_weather_groupby['total_score'] = (
    df_weather_groupby['score_temp'] * 0.3 +
    df_weather_groupby['score_rain_prob'] * 0.2 +
    df_weather_groupby['score_rain_vol'] * 0.3 +
    df_weather_groupby['score_wind'] * 0.1 +
    df_weather_groupby['score_cloud'] * 0.1
)

# 3. Rank the cities on their mean daily score over the whole forecast window,
# so that one perfect day does not outrank five merely good ones.
top_cities = df_weather_groupby.groupby(['city', 'lat', 'lon'])['total_score'].mean().sort_values(ascending=False)
print("--- FINAL RANKING ---")
print(top_cities.head(10))

# Save results
df_weather_groupby.to_csv("weather_forecast_with_score.csv", index=False)

In [ ]:
# Prepare DataFrame for visualization
# The warehouse keeps all 35 cities; only the map is narrowed to the Top-5 the
# deliverable asks for.
df_top_cities = pd.DataFrame(top_cities.reset_index())
df_top_cities_top_5 = df_top_cities.iloc[0:5, :]
df_top_cities_top_5

In [ ]:
# Map 1/2 of the deliverable: the Top-5 destinations by weather score
fig = px.scatter_mapbox(
    df_top_cities_top_5, 
    lat="lat", 
    lon="lon",
    color="total_score",
    size="total_score", 
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15,
    zoom=4, 
    center={"lat": 46.2276, "lon": 2.2137}, # France Center
    mapbox_style="carto-positron", 
    hover_name="city",
    title="Top 5 Destinations in France (Weather Based)"
)

fig.show()

# 3. Accommodation (Booking.com)
We use a Scrapy spider to fetch hotel details for the top cities.

In [ ]:
# Remove previous file if exists
if os.path.exists('hotels.json'):
    os.remove('hotels.json')

# Run the scraper
!cd booking_scraper_project && scrapy crawl booking_spider -O ../hotels.json

## Hotel Analysis & Visualization

In [ ]:
# Load scraped hotel data
df_hotels = pd.read_json("hotels.json")

# Booking hides the review score of properties that have too few reviews; those
# rows cannot be ranked, so they are dropped rather than scored as zero.
df_hotels = df_hotels.dropna()

# Scores are scraped from a French page, where the decimal separator is a comma.
df_hotels['score'] = df_hotels['score'].astype(str).str.replace(',', '.').astype(float)
df_hotels = df_hotels.sort_values(by=["score"], ascending=False)

display(df_hotels.head())

In [ ]:
# Map 2/2 of the deliverable: the Top-20 hotels "in the area", i.e. among the
# Top-5 destinations only -- not the best-rated hotels of the whole country,
# which would sit hundreds of kilometres from the cities recommended above.
df_hotels_top_area = df_hotels[df_hotels['city'].isin(df_top_cities_top_5['city'])]

# df_hotels is already sorted by score, so head(20) is the Top-20 of that area.
fig = px.scatter_mapbox(
    df_hotels_top_area.head(20), 
    lat="lat", 
    lon="lng",
    color="score",
    size="score", 
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15,
    zoom=4, 
    center={"lat": 46.2276, "lon": 2.2137}, 
    mapbox_style="carto-positron", 
    hover_name="name",
    title="Top 20 Hotels in the 5 Best Destinations"
)
fig.show()

# 4. Data Storage (S3 & SQL)
Prepare the dataframes and upload them to the cloud for persistence.

In [ ]:
# Data Formatting
# Assign UUIDs to cities and flatten the weather table for SQL compatibility

# 1. Add IDs to Cities
df_top_cities['id'] = [str(uuid.uuid4()) for _ in range(len(df_top_cities))]

# Save City Table
if not os.path.exists("final_output"):
    os.makedirs("final_output")
    
df_top_cities.rename(columns={'city': 'name'}).to_csv("final_output/villes_table.csv", index=False)

# 2. Flatten Weather Data (MultiIndex -> Single Level)
df_weather_flat = df_weather_groupby.reset_index()
df_weather_flat.columns = ['_'.join(c).strip('_') for c in df_weather_flat.columns.to_flat_index()]

# columns to keep
columns_weather = ['city', 'date', 'temp_mean', 'temp_min', 'temp_max', 
                   'prob_rain_max', 'volume_rain_mean', 'volume_rain_max', 'volume_rain_sum', 
                   'wind_speed_max', 'perc_cloud_mean', 'score_temp', 'score_rain_prob', 
                   'score_rain_vol', 'score_wind', 'score_cloud', 'total_score']
df_weather_flat = df_weather_flat[columns_weather]

# 3. Merge Function for ID linking
def merge_and_save(df_data, df_cities, filename):
    merged = df_data.merge(df_cities[['city', 'id']], on='city', how='left')
    merged = merged.rename(columns={'id': 'city_id'}).drop(columns=['city'])
    merged['id'] = [str(uuid.uuid4()) for _ in range(len(merged))]
    merged.to_csv(f"final_output/{filename}.csv", index=False)
    return merged

df_top_cities_light = df_top_cities[['id', 'city']]
df_weather_final = merge_and_save(df_weather_flat, df_top_cities_light, "weather_table")

if 'df_hotels' in locals():
    df_hotel_final = merge_and_save(df_hotels, df_top_cities_light, "hotels_table")

In [ ]:
# Load the cleaned CSVs into the data lake.
#
# Two arguments boto3 would otherwise infer from AWS have to be given by hand:
# the endpoint, which redirects the calls to R2, and a region -- R2 has none,
# but botocore signs every request with one, and "auto" is what R2 expects.
#
# No try/except on purpose. A silent upload failure leaves the lake out of sync
# with the warehouse, and a printed warning scrolls out of sight in a notebook:
# the ETL has to stop where it breaks.
s3 = boto3.client(
    "s3",
    endpoint_url=r2_endpoint_url,
    aws_access_key_id=r2_access_key_id,
    aws_secret_access_key=r2_secret_access_key,
    region_name="auto",
)

# The bucket is NOT created here. It is infrastructure, created once in the
# Cloudflare console, and the API token is deliberately scoped to it with
# object-level rights only -- so create_bucket() would fail, by design.
files = [f for f in os.listdir("final_output") if f.endswith('.csv')]
for file in files:
    s3.upload_file(f"final_output/{file}", bucket_name, file)
    print(f"Uploaded {file} to R2.")

In [ ]:
# Open the connection to the data warehouse.
#
# sslmode=require: the managed PostgreSQL refuses cleartext connections anyway,
# but psycopg2 defaults to "prefer", which silently falls back to plaintext if
# the handshake fails. Being explicit is what turns the encryption into a
# guarantee rather than a hope.
#
# The port is spelled out rather than left to the 5432 default, so that moving
# the warehouse only ever means editing .env.
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}?sslmode=require"
engine = create_engine(connection_string, echo=False)
meta = MetaData()

In [ ]:
# 1. Cities Table
cities = Table(
    'cities', meta,
    Column('id', String, primary_key=True),
    Column('name', String),
    Column('lat', Float),
    Column('lon', Float),
    Column('total_score', Float)
)

# Idempotent: CREATE TABLE is only issued for what is missing, so re-running the
# notebook against a live warehouse is safe.
meta.create_all(engine)

# The warehouse mirrors the data lake, which overwrites its CSVs on every run.
# Without this the rows would pile up instead of being refreshed -- the UUIDs are
# regenerated each time, so no primary key would ever collide to stop it.
# CASCADE reaches weather and hotels, which carry a foreign key to cities; on a
# first run they do not exist yet and it is simply a no-op.
with engine.begin() as conn:
    conn.execute(text("TRUNCATE cities CASCADE"))

# The schema is owned by the Table() definitions above, not by pandas -- which
# would otherwise infer its own column types on the first write.
df_top_cities.rename(columns={'city': 'name'}).to_sql('cities', engine, if_exists='append', index=False)
print("Cities uploaded to SQL.")

In [ ]:
# 2. Weather Table
weather = Table(
    'weather', meta,
    Column('id', String, primary_key=True),
    Column('date', Date),
    Column('temp_mean', Float),
    Column('temp_min', Float),
    Column('temp_max', Float),
    Column('prob_rain_max', Float),
    Column('volume_rain_mean', Float),
    Column('volume_rain_max', Float),
    Column('volume_rain_sum', Float),
    Column('wind_speed_max', Float),
    Column('perc_cloud_mean', Float),
    Column('score_temp', Float),
    Column('score_rain_prob', Float),
    Column('score_rain_vol', Float),
    Column('score_wind', Float),
    Column('score_cloud', Float),
    Column('total_score', Float),
    Column('city_id', String, ForeignKey('cities.id'))
)

meta.create_all(engine)

# The CSV checkpoint reads dates back as plain strings; the column is a DATE.
df_weather_final['date'] = pd.to_datetime(df_weather_final['date'])
df_weather_final.to_sql('weather', engine, if_exists='append', index=False)
print("Weather uploaded to SQL.")

In [ ]:
# 3. Hotels Table
hotels = Table(
    'hotels', meta,
    Column('id', String, primary_key=True),
    Column('name', String),
    Column('url', String),
    Column('score', Float),
    Column('lat', Float),
    Column('lng', Float),
    Column('description', String),
    Column('city_id', String, ForeignKey('cities.id'))
)

meta.create_all(engine)
df_hotel_final.to_sql('hotels', engine, if_exists='append', index=False)
print("Hotels uploaded to SQL.")